Cell 1 — Markdown

# RetailOps AI
## Notebook 04 — MCP and Human Approval

This notebook adds:

- Retail tools exposed through Model Context Protocol (MCP).
- A state-changing cancellation tool.
- Explicit human approval before any write action.
- A multi-step agent flow using both local and MCP tools.

The original τ²-Bench database remains unchanged. Write operations use a separate working copy.

Cell 2 — Setup

In [ ]:
!pip -q install "mcp>=2.3" openai rank-bm25

In [ ]:
from pathlib import Path
import subprocess
import shutil
import json
import os
import sys
import re

TAU2_DIR = Path("/content/tau2-bench")

if not TAU2_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/sierra-research/tau2-bench.git",
            str(TAU2_DIR)
        ],
        check=True
    )

SDAIA_DIR = Path("/content/agentic-ai")

if not SDAIA_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/SDAIAAcademy/agentic-ai.git",
            str(SDAIA_DIR)
        ],
        check=True
    )

sys.path.insert(0, str(SDAIA_DIR / "labs"))

import agentkit as K

RETAIL_DIR = TAU2_DIR / "data" / "tau2" / "domains" / "retail"
SOURCE_DB = RETAIL_DIR / "db.json"
POLICY_PATH = RETAIL_DIR / "policy.md"

WORK_DIR = Path("/content/RetailOps_AI")
WORK_DIR.mkdir(exist_ok=True)

WORK_DB = WORK_DIR / "retail_working_db.json"

shutil.copy2(SOURCE_DB, WORK_DB)

print(" Environment ready.")
print(" Working database created.")

 Environment ready.
 Working database created.


Cell 3 — Create MCP Server

In [ ]:
MCP_SERVER = WORK_DIR / "retail_mcp.py"

server_code = f'''
# /// script
# requires-python = ">=3.11"
# dependencies = ["mcp>=2.3"]
# ///

import json
from pathlib import Path
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

DB_PATH = Path(r"{WORK_DB}")

server = MCPServer(
    "retailops",
    title="RetailOps MCP Server",
    instructions="Retail order tools using a synthetic benchmark database.",
    version="1.0.0",
    log_level="WARNING",
)

def load_db():
    with open(DB_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_db(db):
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


@server.tool()
def get_order_details(order_id: str, user_id: str) -> dict:
    """Get an order only when it belongs to the identified customer."""

    db = load_db()

    order = db["orders"].get(order_id)

    if not order:
        raise ToolError("Order not found")

    if order["user_id"] != user_id:
        raise ToolError("Order does not belong to this customer")

    return {{
        "order_id": order_id,
        "user_id": order["user_id"],
        "status": order["status"],
        "items": order["items"],
        "cancel_reason": order.get("cancel_reason")
    }}


@server.tool()
def cancel_pending_order(
    order_id: str,
    user_id: str,
    reason: str
) -> dict:
    """Cancel a pending order after host-side human approval."""

    db = load_db()

    order = db["orders"].get(order_id)

    if not order:
        raise ToolError("Order not found")

    if order["user_id"] != user_id:
        raise ToolError("Order does not belong to this customer")

    if order["status"] != "pending":
        raise ToolError("Only pending orders can be cancelled")

    if reason not in ["no longer needed", "ordered by mistake"]:
        raise ToolError("Invalid cancellation reason")

    order["status"] = "cancelled"
    order["cancel_reason"] = reason

    save_db(db)

    return {{
        "order_id": order_id,
        "status": "cancelled",
        "cancel_reason": reason
    }}


if __name__ == "__main__":
    server.run()
'''

MCP_SERVER.write_text(server_code, encoding="utf-8")

print(" MCP server created:")
print(MCP_SERVER)

 MCP server created:
/content/RetailOps_AI/retail_mcp.py


In [ ]:
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from pathlib import Path
import shutil

uv_path = shutil.which("uv")

assert uv_path is not None, "uv is not installed"

mcp_params = StdioServerParameters(
    command=uv_path,
    args=[
        "run",
        "--quiet",
        str(MCP_SERVER)
    ]
)

# Colab's sys.stderr does not support fileno().
# Use a real file for MCP subprocess stderr instead.
MCP_LOG_PATH = Path("/content/retail_mcp_stderr.log")
MCP_ERRLOG = open(MCP_LOG_PATH, "w", encoding="utf-8")

transport = stdio_client(
    mcp_params,
    errlog=MCP_ERRLOG
)

mcp = K.MCPBridge(transport).open()

mcp_tools = mcp.list_tools()

print("✅ MCP connection established.")
print("\nMCP TOOLS:")

for tool in mcp_tools:
    print("•", tool.name)

✅ MCP connection established.

MCP TOOLS:
• get_order_details
• cancel_pending_order


Cell 4 — Connect to MCP

Cell 5 — Load Local Retail Context

In [ ]:
with open(WORK_DB, "r", encoding="utf-8") as f:
    retail_db = json.load(f)

policy_text = POLICY_PATH.read_text(encoding="utf-8")


def find_user_by_email(email: str):

    email = email.strip().lower()

    for user_id, user in retail_db["users"].items():

        if user["email"].lower() == email:

            return {
                "user_id": user_id,
                "name": user["name"],
                "orders": user["orders"]
            }

    return {"error": "User not found"}

Cell 6 — Human Approval Gate

In [ ]:
WRITE_TOOLS = {
    "cancel_pending_order"
}


def request_approval(tool_name, arguments):

    print("\n🛡️ HUMAN APPROVAL REQUIRED")
    print("Tool:", tool_name)
    print("Arguments:")
    print(json.dumps(
        arguments,
        indent=2,
        ensure_ascii=False
    ))

    decision = input(
        "\nApprove this write action? [y/N]: "
    ).strip().lower()

    return decision == "y"

Cell 7 — MCP Tool Executor

In [ ]:
def execute_mcp_tool(name, arguments):

    if name in WRITE_TOOLS:

        approved = request_approval(
            name,
            arguments
        )

        if not approved:

            return {
                "status": "rejected",
                "message": "Human rejected the write action."
            }

    result = mcp.call(
        name,
        arguments
    )

    if result.is_error:
        return {
            "error": result.content[0].text
        }

    text = result.content[0].text

    try:
        return json.loads(text)
    except:
        return {"result": text}

Cell 8 — Select a Real Pending Order

In [ ]:
with open(WORK_DB, "r", encoding="utf-8") as f:
    retail_db = json.load(f)

pending_order_id = next(
    order_id
    for order_id, order in retail_db["orders"].items()
    if order["status"] == "pending"
)

pending_order = retail_db["orders"][pending_order_id]

pending_user_id = pending_order["user_id"]

pending_user = retail_db["users"][pending_user_id]

pending_email = pending_user["email"]

print("Email:", pending_email)
print("Order:", pending_order_id)
print("Current status:", pending_order["status"])

Email: sofia.rossi2645@example.com
Order: #W5918442
Current status: pending


Cell 9 — Read Through MCP

In [ ]:
order_result = execute_mcp_tool(
    "get_order_details",
    {
        "order_id": pending_order_id,
        "user_id": pending_user_id
    }
)

order_result

{'order_id': '#W5918442',
 'user_id': 'sofia_rossi_8776',
 'status': 'pending',
 'items': [{'name': 'Perfume',
   'product_id': '6858788497',
   'item_id': '1725100896',
   'price': 289.66,
   'options': {'scent family': 'oriental',
    'size': '30ml',
    'gender': 'unisex'}},
  {'name': 'Skateboard',
   'product_id': '1968349452',
   'item_id': '5312063289',
   'price': 195.15,
   'options': {'deck material': 'bamboo',
    'length': '31 inch',
    'design': 'graphic'}},
  {'name': 'Action Camera',
   'product_id': '3377618313',
   'item_id': '1586641416',
   'price': 497.39,
   'options': {'resolution': '5K', 'waterproof': 'yes', 'color': 'silver'}},
  {'name': 'Action Camera',
   'product_id': '3377618313',
   'item_id': '6117189161',
   'price': 481.5,
   'options': {'resolution': '4K', 'waterproof': 'yes', 'color': 'silver'}}],
 'cancel_reason': None}

Cell 10 — Write Action + Human Approval

In [ ]:
cancel_result = execute_mcp_tool(
    "cancel_pending_order",
    {
        "order_id": pending_order_id,
        "user_id": pending_user_id,
        "reason": "no longer needed"
    }
)

cancel_result


🛡️ HUMAN APPROVAL REQUIRED
Tool: cancel_pending_order
Arguments:
{
  "order_id": "#W5918442",
  "user_id": "sofia_rossi_8776",
  "reason": "no longer needed"
}

Approve this write action? [y/N]: y


{'order_id': '#W5918442',
 'status': 'cancelled',
 'cancel_reason': 'no longer needed'}

Cell 11 — Verify the Database Changed

In [ ]:
verification = execute_mcp_tool(
    "get_order_details",
    {
        "order_id": pending_order_id,
        "user_id": pending_user_id
    }
)

verification

{'order_id': '#W5918442',
 'user_id': 'sofia_rossi_8776',
 'status': 'cancelled',
 'items': [{'name': 'Perfume',
   'product_id': '6858788497',
   'item_id': '1725100896',
   'price': 289.66,
   'options': {'scent family': 'oriental',
    'size': '30ml',
    'gender': 'unisex'}},
  {'name': 'Skateboard',
   'product_id': '1968349452',
   'item_id': '5312063289',
   'price': 195.15,
   'options': {'deck material': 'bamboo',
    'length': '31 inch',
    'design': 'graphic'}},
  {'name': 'Action Camera',
   'product_id': '3377618313',
   'item_id': '1586641416',
   'price': 497.39,
   'options': {'resolution': '5K', 'waterproof': 'yes', 'color': 'silver'}},
  {'name': 'Action Camera',
   'product_id': '3377618313',
   'item_id': '6117189161',
   'price': 481.5,
   'options': {'resolution': '4K', 'waterproof': 'yes', 'color': 'silver'}}],
 'cancel_reason': 'no longer needed'}

Cell 12 — Rejection Demonstration

In [ ]:
with open(WORK_DB, "r", encoding="utf-8") as f:
    current_db = json.load(f)

second_order_id = next(
    order_id
    for order_id, order in current_db["orders"].items()
    if order["status"] == "pending"
    and order_id != pending_order_id
)

second_user_id = current_db["orders"][second_order_id]["user_id"]

print("Second pending order:", second_order_id)

Second pending order: #W2974929


In [ ]:
rejected_result = execute_mcp_tool(
    "cancel_pending_order",
    {
        "order_id": second_order_id,
        "user_id": second_user_id,
        "reason": "ordered by mistake"
    }
)

rejected_result


🛡️ HUMAN APPROVAL REQUIRED
Tool: cancel_pending_order
Arguments:
{
  "order_id": "#W2974929",
  "user_id": "fatima_anderson_2157",
  "reason": "ordered by mistake"
}

Approve this write action? [y/N]: n


{'status': 'rejected', 'message': 'Human rejected the write action.'}

Cell 13 — Verify Rejected Write Did Not Execute

In [ ]:
rejected_verification = execute_mcp_tool(
    "get_order_details",
    {
        "order_id": second_order_id,
        "user_id": second_user_id
    }
)

print(
    "Order status after rejection:",
    rejected_verification["status"]
)

assert rejected_verification["status"] == "pending"

print("✅ Rejected write action was not executed.")

Order status after rejection: pending
✅ Rejected write action was not executed.


## Conclusion

RetailOps AI now includes a working MCP integration and an enforced human approval gate.

The notebook demonstrated that:

- retail tools are exposed through MCP over stdio,
- the system can read order state through MCP,
- write actions require explicit human approval,
- approved actions change the working database,
- rejected actions do not execute.

The original τ²-Bench database remains unchanged.

### Next

Notebook 05 will add the remaining harness controls:

- AGENTS.md context specification,
- context compaction,
- step budget,
- cost/token budget,
- audit logging.